In [10]:

import gdown
import pandas as pd

df=pd.read_csv("/content/nlp_dataset (1).csv")
df
df = pd.read_csv(output)
display(df.head())

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [11]:

import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    cleaned = [w for w in words if w not in stop_words]
    return " ".join(cleaned)

text_col = 'Text' if 'Text' in df.columns else df.columns[0]
label_col = 'Emotion' if 'Emotion' in df.columns else df.columns[1]

df['cleaned_text'] = df[text_col].apply(clean_text)
display(df[['cleaned_text', label_col]].head())


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,cleaned_text,Emotion
0,seriously hate one subject death feel reluctan...,fear
1,im full life feel appalled,anger
2,sit write start dig feelings think afraid acce...,fear
3,ive really angry r feel like idiot trusting fi...,joy
4,feel suspicious one outside like rapture happe...,fear


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X = df['cleaned_text']
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=5000)

X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (4749, 5000)
Testing data: (1188, 5000)


In [13]:
from sklearn.metrics import accuracy_score, f1_score

results = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM'],
    'Accuracy': [
        accuracy_score(y_test, nb_preds),
        accuracy_score(y_test, svm_preds)
    ],
    'F1 Score': [
        f1_score(y_test, nb_preds, average='weighted'),
        f1_score(y_test, svm_preds, average='weighted')
    ]
})

display(results)

,Model,Accuracy,F1 Score
0,Naive Bayes,0.898148,0.898175
1,SVM,0.939394,0.939365


In [14]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Initialize and train Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_preds = nb_model.predict(X_test)

# Initialize and train SVM
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)

print("Models trained and predictions generated successfully.")

Models trained and predictions generated successfully.


In [15]:
from sklearn.metrics import classification_report

print("Naive Bayes Results:")
print(classification_report(y_test, nb_preds))

print("SVM Results:")
print(classification_report(y_test, svm_preds))

Naive Bayes Results:
              precision    recall  f1-score   support

       anger       0.88      0.91      0.90       400
        fear       0.89      0.91      0.90       388
         joy       0.92      0.88      0.90       400

    accuracy                           0.90      1188
   macro avg       0.90      0.90      0.90      1188
weighted avg       0.90      0.90      0.90      1188

SVM Results:
              precision    recall  f1-score   support

       anger       0.95      0.93      0.94       400
        fear       0.94      0.94      0.94       388
         joy       0.93      0.95      0.94       400

    accuracy                           0.94      1188
   macro avg       0.94      0.94      0.94      1188
weighted avg       0.94      0.94      0.94      1188

